# Prepare additive draft-weight model data

This notebook prepares clean draft features and transaction context. The additive player-production target is constructed after the chronological split in the final modeling stage.


In [1]:
# Set the processed-data location and load analysis libraries.
from pathlib import Path
import json
import numpy as np
import pandas as pd

processed_data_path = Path("../data/processed")

In [2]:
# Load the additive transaction and draft features.
trade_compensation_data = pd.read_parquet(processed_data_path / "transaction_draft_features_additive.parquet")

print("Rows loaded:", len(trade_compensation_data))

Rows loaded: 2749


## Identify the draft tiers and create net counts

In [3]:
# Identify acquired and relinquished draft-tier columns.
acquired_draft_tier_columns = [
    column
    for column in trade_compensation_data.columns
    if (
        column.startswith("acquired_")
        and column.endswith("_count")
        and column != "acquired_total_draft_asset_count"
        and not column.endswith("listed_player_count")
        and not column.endswith("calculated_player_count")
        and not column.endswith("player_id_missing_count")
        and not column.endswith("unresolved_name_count")
        and not column.endswith("no_prior_appearance_count")
        and not column.endswith("no_prior_nba_appearance_count")
    )
]

relinquished_draft_tier_columns = [
    column
    for column in trade_compensation_data.columns
    if (
        column.startswith("relinquished_")
        and column.endswith("_count")
        and column != "relinquished_total_draft_asset_count"
        and not column.endswith("listed_player_count")
        and not column.endswith("calculated_player_count")
        and not column.endswith("player_id_missing_count")
        and not column.endswith("unresolved_name_count")
        and not column.endswith("no_prior_appearance_count")
        and not column.endswith("no_prior_nba_appearance_count")
    )
]

acquired_tiers = {column.removeprefix("acquired_").removesuffix("_count") for column in acquired_draft_tier_columns}
relinquished_tiers = {column.removeprefix("relinquished_").removesuffix("_count") for column in relinquished_draft_tier_columns}

if acquired_tiers != relinquished_tiers:
    raise ValueError("Acquired and relinquished draft tiers do not match.")

draft_tiers = sorted(acquired_tiers)

for tier in draft_tiers:
    trade_compensation_data[f"net_{tier}_count"] = (
        trade_compensation_data[f"acquired_{tier}_count"] - trade_compensation_data[f"relinquished_{tier}_count"]
    )

## Exclude draft assets that cannot be valued

In [4]:
# Remove unknown or unranked draft terms from model inputs.
excluded_tier_terms = ("unknown", "unavailable", "unranked")

excluded_draft_tiers = sorted([tier for tier in draft_tiers if any(term in tier for term in excluded_tier_terms)])

excluded_draft_count_columns = [f"{side}_{tier}_count" for side in ["acquired", "relinquished"] for tier in excluded_draft_tiers]

trade_compensation_data["excluded_draft_asset_count"] = trade_compensation_data[excluded_draft_count_columns].sum(axis=1)

trade_compensation_data["has_excluded_draft_asset"] = trade_compensation_data["excluded_draft_asset_count"].gt(0)

draft_weight_model_data = trade_compensation_data.loc[~trade_compensation_data["has_excluded_draft_asset"]].copy().reset_index(drop=True)

print("Rows before exclusion:", len(trade_compensation_data))
print("Rows after exclusion:", len(draft_weight_model_data))

Rows before exclusion: 2749
Rows after exclusion: 1963


## Define estimable outright-pick and swap features

In [5]:
# Define the ordered first- and second-round value hierarchy.
outright_pick_hierarchy = [
    "projected_late_second",
    "projected_early_second",
    "projected_late_first",
    "projected_lottery_first",
    "projected_top_5_first",
]

swap_draft_tiers = [
    "projected_late_second_swap",
    "projected_early_second_swap",
    "projected_late_first_swap",
    "projected_lottery_first_swap",
    "projected_top_5_first_swap",
]

outright_pick_net_columns = [f"net_{tier}_count" for tier in outright_pick_hierarchy]

swap_net_columns = [f"net_{tier}_count" for tier in swap_draft_tiers]

required_model_columns = [*outright_pick_net_columns, *swap_net_columns]

missing_model_columns = [column for column in required_model_columns if column not in draft_weight_model_data.columns]

if missing_model_columns:
    raise KeyError(f"Missing model columns: {missing_model_columns}")

## Retain transaction identifiers, coverage fields, and draft features

In [6]:
# Select identifiers, production measures, targets, and draft terms.
identifier_columns = [
    column
    for column in ["transaction_row_id", "Date", "Team", "Season", "Acquired", "Relinquished", "Notes"]
    if column in draft_weight_model_data.columns
]

coverage_columns = [
    column
    for column in [
        "acquired_listed_player_count",
        "relinquished_listed_player_count",
        "acquired_calculated_player_count",
        "relinquished_calculated_player_count",
        "acquired_player_id_missing_count",
        "relinquished_player_id_missing_count",
        "acquired_unresolved_name_count",
        "relinquished_unresolved_name_count",
        "acquired_no_prior_appearance_count",
        "relinquished_no_prior_appearance_count",
        "acquired_no_prior_nba_appearance_count",
        "relinquished_no_prior_nba_appearance_count",
        "asset_only_transaction",
    ]
    if column in draft_weight_model_data.columns
]

draft_count_columns = [
    column
    for column in draft_weight_model_data.columns
    if (
        column.startswith(("acquired_", "relinquished_", "net_"))
        and column.endswith("_count")
        and (
            any(tier in column for tier in draft_tiers)
            or column in {"acquired_total_draft_asset_count", "relinquished_total_draft_asset_count"}
        )
    )
]

model_output_columns = list(
    dict.fromkeys([*identifier_columns, *coverage_columns, *draft_count_columns, "excluded_draft_asset_count", "has_excluded_draft_asset"])
)

draft_weight_model_data = draft_weight_model_data[model_output_columns].copy().reset_index(drop=True)

## Save the model input and configuration

In [7]:
# Save the model-ready table and its configuration.
model_output_path = processed_data_path / "draft_weight_model_data_additive.parquet"

draft_weight_model_data.to_parquet(model_output_path, index=False)

draft_weight_config = {
    "raw_target": "net_player_production_value",
    "model_target": "draft_weight_target",
    "sign_convention": (
        "Positive net draft count means draft compensation received. "
        "Positive player-value differential means more player production acquired."
    ),
    "outright_pick_hierarchy": outright_pick_hierarchy,
    "swap_draft_tiers": swap_draft_tiers,
    "excluded_draft_tiers": excluded_draft_tiers,
    "fixed_zero_draft_tiers": ["zero_value_later_round"],
    "unit_of_analysis": "team_trade_perspective",
    "row_weighting": "equal_weight_per_team_perspective",
    "player_value_eligibility": (
        "Both player sides must be fully resolved. Empty player sides are valued at zero. "
        "Known no-prior-NBA-appearance players are valued at zero."
    ),
}

config_output_path = processed_data_path / "draft_weight_model_config_additive.json"

with open(config_output_path, "w", encoding="utf-8") as file:
    json.dump(draft_weight_config, file, indent=2)

print("Saved:", model_output_path)
print("Saved:", config_output_path)

Saved: ..\data\processed\draft_weight_model_data_additive.parquet
Saved: ..\data\processed\draft_weight_model_config_additive.json
